# Working with Franka Robotics FR3 through ROS 2

This notebook introduces the ROS 2 workflow for the Franka FR3 robot in RobotBlockSet. Its purpose is to show how to connect to a running `franka_ros2` system, inspect robot state, switch controllers, and send joint-space and Cartesian commands through the common RobotBlockSet API.


## What this notebook covers

The examples below demonstrate the main steps for working with an FR3 through ROS 2:

- connecting to a running ROS 2 robot or simulation,
- checking available RobotBlockSet control strategies,
- executing joint trajectory and Cartesian motions,
- adjusting Cartesian impedance parameters,
- updating TCP and load settings,
- stopping motion and shutting down cleanly.

Use this notebook as a practical starting point after the ROS 2 bringup has already been launched in another terminal.

# ROS 2 bringup

Before creating the RobotBlockSet object, start a ROS 2 system that provides the following interfaces:

| Interface | Used by |
| --- | --- |
| `/joint_states` | robot state feedback |
| `/controller_manager/*` services | controller loading and switching |
| `fr3_arm_controller/follow_joint_trajectory` (action interface) | `JointPositionTrajectory` strategy  |
| `fr3_arm_controller/joint_trajectory` (topic interface) | `JointPosition` |
| `cartesian_impedance_controller/cartesian_command` | `CartesianImpedance` strategy |
| `/franka_robot_state_broadcaster/external_wrench_in_base_frame` | external wrench feedback on hardware |

For the Gazebo setup in this workspace, one suitable launch command is:

```bash
ros2 launch compliant_controllers fr3_gz.launch.py load_gripper:=true
```

For hardware, use the matching Franka bringup and controller configuration for your robot. When you use a ROS namespace, pass the same namespace to `fr3(ns=...)` below.

# Imports

In [ ]:
import time
import numpy as np
import rclpy

from robotblockset.ros2.franka_ros2 import fr3
from robotblockset.ros2.grippers_ros2 import FrankaGripper
from robotblockset.transformations import map_pose, rot_z, x2t, rp2t

np.set_printoptions(formatter={"float": "{: 0.4f}".format})

# Franka ROS 2 robot class

Class `fr3` in `robotblockset.ros2.franka_ros2` is a ROS 2-backed RobotBlockSet robot class for Franka FR3. It combines the generic robot API with ROS 2 controller interfaces and the FR3 kinematic specification.

| Init argument | Description |
| --- | --- |
| `name` | Robot name and ROS 2 node name. Default is `"fr3"`. It is also used to build joint names such as `fr3_joint1`. |
| `ns` | ROS namespace for topics, actions, and services. Use `""` for non-namespaced bringup. |
| `control_strategy` | Initial RobotBlockSet strategy. Supported values are `"CartesianImpedance"`, `"JointPositionTrajectory"`, and `"JointPosition"`. |
| `SIM` | Set to `True` for simulation. Hardware-only services such as `SetLoad` and robot-side TCP updates are skipped in simulation. |

Compared to the base `robot_ros2` class, `fr3` adds Franka-specific controller names, FR3 joint naming, the FR3 kinematic model, external wrench subscription, and Franka services for load and TCP configuration on hardware.

# Connect to the robot

Create the ROS 2 client object after the robot or simulation is already running. The constructor starts a background executor, waits briefly for joint states, and activates the requested controller strategy.

Use `SIM=True` for Gazebo or fake hardware. Use `SIM=False` on a physical FR3 so that load, TCP frame, and wrench services are enabled.

In [ ]:
if not rclpy.ok():
    rclpy.init()

r = fr3(control_strategy="JointPositionTrajectory", SIM=True)

For a namespaced robot, pass the namespace without leading or trailing slashes:

```python
r = fr3(ns="franka1", control_strategy="JointPositionTrajectory", SIM=True)
```

The wrapper then subscribes to `/franka1/joint_states` and uses the controller manager and controllers in the same namespace.

# Robot state

The latest ROS 2 joint state message is copied into the RobotBlockSet state container with `GetState()` or `Update()`. The Cartesian pose and velocity are computed from the FR3 kinematic model.

| Property | Description |
| --- | --- |
| `r.q` | Actual joint positions. |
| `r.qdot` | Actual joint velocities. |
| `r.trq` | Actual joint efforts from `/joint_states`. |
| `r.x` | Actual TCP pose in RobotBlockSet compact pose format. |
| `r.p`, `r.R`, `r.T` | Position, rotation matrix, and homogeneous transform. |
| `r.FT` | External wrench when the Franka state broadcaster is available. |
| `r.q_ref`, `r.x_ref` | Current commanded joint and Cartesian targets. |

In [ ]:
r.GetState()

print("Connected:", r.isConnected())
print("q:", r.q)
print("x:", r.x)
print("T:")
print(r.T)

# Control strategies

`fr3` exposes ROS 2 controllers as RobotBlockSet control strategies:

| Strategy | ROS 2 controller interface | Typical use |
| --- | --- | --- |
| `JointPositionTrajectory` | `fr3_arm_controller/follow_joint_trajectory` action | Time-parameterized joint moves. |
| `JointPosition` | `fr3_arm_controller/joint_trajectory` topic | Direct joint-position setpoints. |
| `CartesianImpedance` | `cartesian_impedance_controller/cartesian_command` topic | Cartesian pose commands with compliance parameters. |

`SetStrategy(...)` switches the active ROS 2 controller through the controller manager and resets the current RobotBlockSet target to the measured state before changing interfaces.

In [ ]:
print(r.AvailableStrategies())

# Joint-space motion

The FR3 ROS 2 wrapper supports two joint-space strategies:

| Strategy | How commands are sent | When to use it |
| --- | --- | --- |
| `JointPosition` | Publishes joint points to `fr3_arm_controller/joint_trajectory`. | Use when you want point-by-point command updates and more direct control over the setpoints sent to the controller. |
| `JointPositionTrajectory` | Sends a full trajectory to the `follow_joint_trajectory` action server. | Use for regular timed moves where the action server should execute and report the complete trajectory. The action result can be checked with `WaitUntilDone()`. |


In both startegies, you can use `JMove` and `JMoveFor`.

In [ ]:
r.SetStrategy("JointPosition")
q_target = [0.5, -0.2, -0.6, -2.1, -0.2, 1.9, 0.79]
start_time = time.time()
r.JMove(q_target, t=2.0)

In [ ]:
r.SetStrategy("JointPositionTrajectory")
r.EnableWaitForActionServer(True)

r.JMove(q_target, 2.0)
result = r.WaitUntilDone(timeout=5.0)
print("Result:", r.MotionResultStr(result))


`JMoveFor(...)` moves relative to the current commanded joint target. This is useful for small checks after a known starting pose.

In [ ]:
r.JMoveFor([0.1, 0, 0, 0, 0, 0, 0], 2.0)
result = r.WaitUntilDone(timeout=5.0)
print("Result:", r.MotionResultStr(result))

You can also use inverse kinematics to command robot in Cartesian space, while the robot uses joint-space control under the hood. For example, to move the robot 10 cm in the z direction:

In [ ]:
r.CMoveFor([0.0, 0.0, 0.10], t=2.0, max_iterations=25, traj_samp_fac=10)
r.CMoveFor([0.0, 0.0, -0.10], t=2.0, max_iterations=25, traj_samp_fac=10)

# Cartesian impedance strategy

`CartesianImpedance` publishes Cartesian command messages. The target pose is expressed through RobotBlockSet task-space conventions, while the active controller receives pose, velocity, feed-forward wrench, stiffness, damping, and stiffness-frame data.

After switching from a joint controller, it is often useful to refresh state and reset the target to the measured configuration before the first Cartesian command.

In [ ]:
r.SetStrategy("CartesianImpedance")
r.ResetCurrentTarget(do_move=True)
r.SetCartesianStiff()

A small vertical relative move is a good first Cartesian test. In simulation, this command is also useful for checking that the Cartesian impedance controller is receiving messages.

In [ ]:
start = time.time()
r.CMoveFor([0.0, 0.0, 0.10], t=2.0)
r.CMoveFor([0.0, 0.0, -0.10], t=2.0)
print("Execution time:", time.time() - start)
print("x:", r.x)

`CMove(...)` can also accept an absolute pose. The example below keeps the current position and rotates the TCP target around the z axis.

In [ ]:
R_target = rot_z(45, unit="deg", out="R") @ r.R
T_target = map_pose(p=r.p, R=R_target, out="T")

r.CMove(T_target, t=2.0)

## Cartesian compliance

The Cartesian impedance controller can be tuned globally or per command.

| Parameter | Description |
| --- | --- |
| `Kp` | Translational stiffness vector. |
| `Kr` | Rotational stiffness vector. |
| `R` | Rotation matrix for the translational stiffness frame. |
| `D` | Damping scaling factor. |

`SetCartesianCompliance(...)` stores new controller parameters and, by default, republishes the current target so the robot holds pose with the updated compliance.

In [ ]:
R45 = rot_z(45, unit="deg", out="R")

r.SetCartesianCompliance(
    Kp=[1000.0, 0.0, 1000.0],
    Kr=[30.0, 30.0, 30.0],
    R=R45,
    D=2.0,
)

r.CMoveFor([0.0, 0.0, -0.05], t=1.0)
r.CMoveFor([0.0, 0.0, 0.05], t=1.0)

r.SetCartesianStiff()

# TCP

`SetTCP(...)` updates the RobotBlockSet TCP. On hardware, it can also forward the TCP frame to the Franka service server. In simulation, service forwarding is disabled, so use `send_to_robot=False` or keep `SIM=True`.

| Method | Description |
| --- | --- |
| `SetTCP(tcp, frame="Gripper", send_to_robot=True)` | Set the active TCP relative to the gripper or flange. |
| `GetTCP(out="T")` | Return the active TCP in the requested representation. |
| `SetTCPGripper(tcp)` | Set the nominal gripper TCP used when `frame="Gripper"`. |

In [ ]:
r.SetTCP([0.0, 0.0, 0.10], frame="Flange", send_to_robot=False)
print(r.GetTCP(out="T"))

r.SetTCP(send_to_robot=False)

# Load and external wrench

On hardware, `SetLoad(...)` sends the end-effector load to the Franka service server. The wrapper temporarily deactivates the active controller, calls the service, and activates the controller again.

```python
r.SetLoad(mass=0.73, COM=[0.0, 0.0, 0.04])
```

When `SIM=False`, the wrapper also subscribes to `/franka_robot_state_broadcaster/external_wrench_in_base_frame`, and the latest wrench is available through `r.FT` after `GetState()`.

# Franka gripper

`FrankaGripper` is the ROS 2 action-based interface for the Franka gripper. It can be used standalone with `FrankaGripper()`, or it can share the robot node with `FrankaGripper(robot=r)`.

| Method | Description |
| --- | --- |
| `Homing()` | Home the gripper and open it fully. |
| `Move(width, speed=0.1)` | Move to a target opening width in metres. |
| `Grasp(width, speed=0.1, force=5.0, eps=0.005)` | Close around an object with force and tolerance. |
| `Stop()` | Stop the current gripper action. |

The standard ROS 2 gripper launch creates action servers under `/franka_gripper/franka_gripper/...`. If your gripper is namespaced, pass the same namespace to `FrankaGripper(namespace=...)`.

In [ ]:
g = FrankaGripper(robot=r)

print("Homing gripper:", g.Homing())
print("Move to pre-grasp width:", g.Move(width=0.02))
print("Width feedback:", g.width)

# Example: rotate a knob

A knob-turning task is a useful compact example because it combines Cartesian approach, grasping, rotational motion, and release. Replace `p_start` and `Q_start` with the measured pose of your knob before running this on hardware.

The example uses a simple, readable start pose: the TCP is placed in front of the knob, and the tool orientation is represented by a unit quaternion. The approach point is offset by 3 cm from the knob pose, then the robot moves to the knob, grasps it, rotates 45 degrees about the current tool frame, and releases.

In [ ]:
r.SetStrategy("CartesianImpedance")
r.GetState()
r.ResetCurrentTarget()
r.SetCartesianStiff()

# Replace this template pose with the measured knob pose.
p_start = np.array([0.45, 0.00, 0.35])
Q_start = np.array([0.0, 1.0, 0.0, 0.0])
T_start = map_pose(p=p_start, Q=Q_start, out="T")

# Approach, move onto the knob, grasp, turn, and release.
r.CApproach(T_start, [0.0, 0.0, 0.03], t=2.0)
r.CMove(T_start, t=2.0)

g.Grasp(width=0.009, force=10.0)

r.CMoveFor(rot_z(45, unit="deg", out="R"), t=3.0)

# FrankaGripper has no Open() wrapper; opening is a Move to a wider width.
g.Move(width=0.08)

# Status, stopping, and shutdown

`StopMotion()` aborts the active controller motion when supported, calls the generic stop logic, and resets the target to the current state. At the end of a notebook session, shut down both the RobotBlockSet ROS 2 node and `rclpy`.

In [ ]:
r.StopMotion()
r.ResetCurrentTarget()
print("Status:", r.Check())

In [ ]:
r.shutdown()

if rclpy.ok():
    rclpy.shutdown()